# Assignment 2: Word Representations

**Project:** Job Skill Analysis

**Dataset:** Cleaned Job Skill Set Dataset

## Part A: Count-Based Word Representations

This notebook implements Bag of Words, TF-IDF, and N-gram representations using the cleaned corpus from Assignment 1.


In [2]:
import pandas as pd
from pathlib import Path

# Path to the cleaned dataset
dataset_path = Path("../data/processed/cleaned_job_skill_dataset.parquet")

# Load the Parquet dataset
df = pd.read_parquet(dataset_path)

# Display the dataset's dimensions
print("Dataset shape:", df.shape)

# Display column names
print("\nColumns:")
print(df.columns.tolist())

# Show the first five records
display(df.head())

Dataset shape: (1167, 7)

Columns:
['job_id', 'category', 'job_title', 'job_description', 'job_skill_set', 'word_count', 'cleaned_description']


,job_id,category,job_title,job_description,job_skill_set,word_count,cleaned_description
0,3902668440,HR,Sr Human Resource Generalist,SUMMARY\nTHE SR. HR GENERALIST PROVIDES HR EXP...,"['employee relations', 'talent acquisition', '...",583,summary the sr hr generalist provide hr expert...
1,3905823748,HR,Human Resources Manager,BE PART OF A STELLAR TEAM AT YSB AS THE MANAGE...,"['Talent Acquisition', 'Employee Performance M...",323,be part of a stellar team at ysb as the manage...
2,3905854799,HR,Director of Human Resources,OUR CLIENT IS A THRIVING ORGANIZATION OFFERING...,"['Human Resources Management', 'Recruitment', ...",666,our client be a thrive organization offer a un...
3,3905834061,HR,Chief Human Resources Officer,JOB TITLE: CHIEF HUMAN RESOURCES OFFICER (CHRO...,"['talent management', 'organizational developm...",416,job title chief human resource officer chro)in...
4,3906250451,HR,Human Resources Generalist (Hybrid Role),DESCRIPTION\n\n WHO WE ARE \n\nAVI-SPL IS A DI...,"['Microsoft Office', 'Data analysis', 'Employe...",604,description who we be avi spl be a digital ena...


In [3]:
# Select the cleaned text column from Assignment 1
corpus = df["cleaned_description"].fillna("").astype(str)

print("Number of documents:", len(corpus))

print("\nFirst cleaned document:")
print(corpus.iloc[0][:500])

print("\nEmpty documents:", (corpus.str.strip() == "").sum())

Number of documents: 1167

First cleaned document:
summary the sr hr generalist provide hr expertise across various function include employee relation talent acquisition performance management payroll compensation and benefit the sr hr generalist have responsibility that be strategic administrative and consultative this role provide consultative hr business partner support to assign business unit across all aspect of the employee life cycle this role also support hr operation with weekly payroll processing the sr hr generalist be also responsibl

Empty documents: 0


In [4]:
from sklearn.feature_extraction.text import CountVectorizer

# Create the Bag of Words vectorizer
bow_vectorizer = CountVectorizer()

# Learn vocabulary and transform documents into word counts
bow_matrix = bow_vectorizer.fit_transform(corpus)

print("Bag of Words matrix shape:", bow_matrix.shape)
print("Vocabulary size:", len(bow_vectorizer.vocabulary_))

Bag of Words matrix shape: (1167, 18635)
Vocabulary size: 18635


In [5]:
# Total number of cells in the document-term matrix
total_cells = bow_matrix.shape[0] * bow_matrix.shape[1]

# Number of nonzero cells
nonzero_cells = bow_matrix.nnz

# Calculate percentage of zero cells
sparsity = (1 - nonzero_cells / total_cells) * 100

print("Total matrix cells:", total_cells)
print("Nonzero cells:", nonzero_cells)
print(f"Sparsity: {sparsity:.2f}%")

Total matrix cells: 21747045
Nonzero cells: 305409
Sparsity: 98.60%


In [6]:
# Get vocabulary terms
feature_names = bow_vectorizer.get_feature_names_out()

# Display first 5 documents and first 15 vocabulary terms
bow_preview = pd.DataFrame(
    bow_matrix[:5, :15].toarray(),
    columns=feature_names[:15]
)

display(bow_preview)

,00,000,00008,0001099,00072,000annual,000benefits,000first,000for,000full,000location,000on,004119,004405,004956
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [7]:
import numpy as np

# Sum each word's frequency across all documents
word_counts = np.asarray(bow_matrix.sum(axis=0)).ravel()

# Sort words by frequency
top_indices = np.argsort(word_counts)[::-1][:15]

# Display the 15 most frequent words
top_words = pd.DataFrame({
    "Word": feature_names[top_indices],
    "Frequency": word_counts[top_indices]
})

display(top_words)

,Word,Frequency
0,and,39042
1,to,19649
2,the,17308
3,of,13636
4,be,10726
5,in,10357
6,with,9034
7,for,7184
8,or,5245
9,our,4236


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create the TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer()

# Learn vocabulary and calculate TF-IDF weights
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Vocabulary size:", len(tfidf_vectorizer.vocabulary_))

TF-IDF matrix shape: (1167, 18635)
Vocabulary size: 18635


In [9]:
# Calculate total number of matrix cells
total_cells_tfidf = tfidf_matrix.shape[0] * tfidf_matrix.shape[1]

# Count nonzero values
nonzero_tfidf = tfidf_matrix.nnz

# Calculate sparsity
tfidf_sparsity = (1 - nonzero_tfidf / total_cells_tfidf) * 100

print("Total cells:", total_cells_tfidf)
print("Nonzero cells:", nonzero_tfidf)
print(f"TF-IDF sparsity: {tfidf_sparsity:.2f}%")

print(f"BoW sparsity: {sparsity:.2f}%")

Total cells: 21747045
Nonzero cells: 305409
TF-IDF sparsity: 98.60%
BoW sparsity: 98.60%


In [10]:
import numpy as np
import pandas as pd

# Get vocabulary words
tfidf_features = tfidf_vectorizer.get_feature_names_out()

# Select the first job description
doc_index = 0

# Extract TF-IDF weights for this document
doc_vector = tfidf_matrix[doc_index].toarray().ravel()

# Find the 10 highest-weighted words
top_indices = np.argsort(doc_vector)[::-1][:10]

# Create a table
top_tfidf_words = pd.DataFrame({
    "Word": tfidf_features[top_indices],
    "TF-IDF Weight": doc_vector[top_indices]
})

display(top_tfidf_words)

,Word,TF-IDF Weight
0,and,0.473093
1,hr,0.437638
2,payroll,0.228792
3,to,0.175520
4,with,0.168322
5,in,0.150346
6,generalist,0.132724
7,employee,0.126467
8,partner,0.126153
9,of,0.123602


In [11]:
# Get IDF values for all vocabulary terms
idf_values = tfidf_vectorizer.idf_

# Create a table of terms and IDF values
idf_table = pd.DataFrame({
    "Word": tfidf_features,
    "IDF": idf_values
})

# Show 15 words with the lowest IDF
common_words = idf_table.sort_values("IDF").head(15)

display(common_words)

rare_words = idf_table.sort_values("IDF", ascending=False).head(15)

display(rare_words)

,Word,IDF
1499,and,1.003431
17176,to,1.005150
11271,of,1.011193
8261,in,1.012926
16993,the,1.013793
18371,with,1.014662
2247,be,1.026900
6945,for,1.038399
6359,experience,1.088569
11560,or,1.088569


,Word,IDF
18634,ōwner,7.369901
18633,ōninite,7.369901
18632,ōnin,7.369901
2,00008,7.369901
3,0001099,7.369901
4,00072,7.369901
5,000annual,7.369901
18631,état,7.369901
18615,zendesk,7.369901
18614,zealand,7.369901


In [12]:
# Select document 0
doc_index = 0

# Get terms present in the document
present_indices = bow_matrix[doc_index].nonzero()[1]

# Get BoW counts
bow_counts = bow_matrix[doc_index, present_indices].toarray().ravel()

# Get TF-IDF weights
tfidf_weights = tfidf_matrix[doc_index, present_indices].toarray().ravel()

# Create comparison table
comparison_df = pd.DataFrame({
    "Word": bow_vectorizer.get_feature_names_out()[present_indices],
    "BoW Count": bow_counts,
    "TF-IDF Weight": tfidf_weights
})

# Sort by TF-IDF weight
comparison_df = comparison_df.sort_values(
    "TF-IDF Weight",
    ascending=False
)

display(comparison_df.head(15))

,Word,BoW Count,TF-IDF Weight
19,and,54,0.473093
3,hr,20,0.437638
17,payroll,8,0.228792
33,to,20,0.175520
43,with,19,0.168322
76,in,17,0.150346
4,generalist,4,0.132724
11,employee,9,0.126467
31,partner,7,0.126153
38,of,14,0.123602


### TF-IDF Analysis

TF-IDF represents documents by assigning numerical weights to words based on their frequency within a document and their rarity across the corpus.

Unlike Bag of Words, which uses raw word counts, TF-IDF reduces the relative importance of terms that appear across many documents.

The highest-weighted terms in a job description can help identify its distinctive skills, technologies, and responsibilities.

BoW and TF-IDF have the same matrix shape and sparsity when they use the same vocabulary and tokenization rules, but their numerical values are different.

## Part A3: N-gram Representations

N-grams capture sequences of consecutive words.

We compare unigram, unigram + bigram, and unigram + bigram + trigram representations using CountVectorizer and TfidfVectorizer.

The objective is to examine vocabulary growth and the trade-off between richer contextual information and matrix sparsity.

In [13]:
from sklearn.feature_extraction.text import CountVectorizer

# Unigrams only
unigram_vectorizer = CountVectorizer(ngram_range=(1, 1))
unigram_matrix = unigram_vectorizer.fit_transform(corpus)

# Unigrams + bigrams
bigram_vectorizer = CountVectorizer(ngram_range=(1, 2))
bigram_matrix = bigram_vectorizer.fit_transform(corpus)

# Unigrams + bigrams + trigrams
trigram_vectorizer = CountVectorizer(ngram_range=(1, 3))
trigram_matrix = trigram_vectorizer.fit_transform(corpus)

print("Unigram shape:", unigram_matrix.shape)
print("Unigram + Bigram shape:", bigram_matrix.shape)
print("Unigram + Bigram + Trigram shape:", trigram_matrix.shape)

Unigram shape: (1167, 18635)
Unigram + Bigram shape: (1167, 205216)
Unigram + Bigram + Trigram shape: (1167, 597126)


In [14]:
def calculate_sparsity(matrix):
    total_cells = matrix.shape[0] * matrix.shape[1]

    if total_cells == 0:
        return 0.0

    zero_percentage = (1 - matrix.nnz / total_cells) * 100

    return zero_percentage


ngram_results = pd.DataFrame({
    "Representation": [
        "Unigrams (1,1)",
        "Unigrams + Bigrams (1,2)",
        "Unigrams + Bigrams + Trigrams (1,3)"
    ],

    "Vocabulary Size": [
        unigram_matrix.shape[1],
        bigram_matrix.shape[1],
        trigram_matrix.shape[1]
    ],

    "Matrix Shape": [
        str(unigram_matrix.shape),
        str(bigram_matrix.shape),
        str(trigram_matrix.shape)
    ],

    "Sparsity (%)": [
        calculate_sparsity(unigram_matrix),
        calculate_sparsity(bigram_matrix),
        calculate_sparsity(trigram_matrix)
    ]
})

display(ngram_results)

,Representation,Vocabulary Size,Matrix Shape,Sparsity (%)
0,"Unigrams (1,1)",18635,"(1167, 18635)",98.595630
1,"Unigrams + Bigrams (1,2)",205216,"(1167, 205216)",99.642899
2,"Unigrams + Bigrams + Trigrams (1,3)",597126,"(1167, 597126)",99.792982


In [15]:
unigram_vocab_size = unigram_matrix.shape[1]
bigram_vocab_size = bigram_matrix.shape[1]
trigram_vocab_size = trigram_matrix.shape[1]

bigram_growth = (
    (bigram_vocab_size - unigram_vocab_size)
    / unigram_vocab_size
) * 100

trigram_growth = (
    (trigram_vocab_size - unigram_vocab_size)
    / unigram_vocab_size
) * 100

print(f"Vocabulary growth with bigrams: {bigram_growth:.2f}%")
print(f"Vocabulary growth with trigrams: {trigram_growth:.2f}%")

Vocabulary growth with bigrams: 1001.24%
Vocabulary growth with trigrams: 3104.33%


In [16]:
# Get all features from the trigram vectorizer
all_features = trigram_vectorizer.get_feature_names_out()

# Extract phrases containing exactly 2 words
bigrams = [
    feature for feature in all_features
    if len(feature.split()) == 2
]

# Extract phrases containing exactly 3 words
trigrams = [
    feature for feature in all_features
    if len(feature.split()) == 3
]

print("First 20 bigrams:")
print(bigrams[:20])

print("\nFirst 20 trigrams:")
print(trigrams[:20])

First 20 bigrams:
['00 00', '00 100', '00 108', '00 111', '00 114', '00 115', '00 130', '00 144', '00 149', '00 156', '00 159', '00 16', '00 161', '00 164', '00 17', '00 18', '00 19', '00 20', '00 209', '00 21']

First 20 trigrams:
['00 00 est', '00 00 the', '00 100 000', '00 108 900', '00 111 000', '00 111 750', '00 114 663', '00 115 000', '00 130 000', '00 144 900', '00 149 500', '00 156 900', '00 159 390', '00 16 00', '00 16 50', '00 161 000', '00 161 400', '00 164 900', '00 17 25', '00 18 75']


In [17]:
import numpy as np

features = trigram_vectorizer.get_feature_names_out()

frequencies = np.asarray(
    trigram_matrix.sum(axis=0)
).ravel()

phrase_df = pd.DataFrame({
    "Phrase": features,
    "Frequency": frequencies
})

# Count words in each feature
phrase_df["N"] = phrase_df["Phrase"].str.split().str.len()

top_bigrams = (
    phrase_df[phrase_df["N"] == 2]
    .sort_values("Frequency", ascending=False)
    .head(10)
)

top_trigrams = (
    phrase_df[phrase_df["N"] == 3]
    .sort_values("Frequency", ascending=False)
    .head(10)
)

print("Top 10 Bigrams:")
display(top_bigrams)

print("Top 10 Trigrams:")
display(top_trigrams)

Top 10 Bigrams:


,Phrase,Frequency,N
349911,of the,1956,2
262333,in the,1649,2
7763,ability to,1623,2
584666,with the,1080,2
546118,to the,957,2
592488,year of,939,2
250863,human resource,916,2
576784,will be,883,2
200684,experience in,823,2
569565,we be,780,2


Top 10 Trigrams:


,Phrase,Frequency,N
86044,be responsible for,432,3
515813,the ability to,424,3
69483,as well as,404,3
592541,year of experience,338,3
81922,be an equal,334,3
190190,equal opportunity employer,326,3
28362,an equal opportunity,317,3
78304,bachelor degree in,297,3
81584,be able to,262,3
8041,ability to work,245,3


In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_ngram_results = []

for ngram_range in [(1, 1), (1, 2), (1, 3)]:

    vectorizer = TfidfVectorizer(
        ngram_range=ngram_range
    )

    matrix = vectorizer.fit_transform(corpus)

    tfidf_ngram_results.append({
        "N-gram Range": str(ngram_range),
        "Vocabulary Size": matrix.shape[1],
        "Matrix Shape": str(matrix.shape),
        "Sparsity (%)": calculate_sparsity(matrix)
    })

tfidf_ngram_df = pd.DataFrame(tfidf_ngram_results)

display(tfidf_ngram_df)

,N-gram Range,Vocabulary Size,Matrix Shape,Sparsity (%)
0,"(1, 1)",18635,"(1167, 18635)",98.595630
1,"(1, 2)",205216,"(1167, 205216)",99.642899
2,"(1, 3)",597126,"(1167, 597126)",99.792982


### N-gram Analysis

The experiment compares three feature configurations:
unigrams (1,1), unigrams and bigrams (1,2), and
unigrams, bigrams and trigrams (1,3).

Increasing the N-gram range allows the model to capture
multiword expressions, which is useful for job descriptions
containing phrases such as "machine learning" and
"project management".

However, longer N-grams increase vocabulary size and
feature dimensionality. They can also increase sparsity
because many phrases appear in relatively few documents.

The final comparison is based on the measured vocabulary
sizes and sparsity percentages.

(1, 1)	vocab size = 18635 , sparsity percentage = 98.595630
(1, 2)	vocab size = 205216, sparsity percentage = 99.642899
(1, 3)	vocab size = 597126, sparsity percentage = 99.792982


## Part A4: Manual TF-IDF and Bigram Calculations

We use a small corpus of four job-related documents to manually
calculate a TF-IDF weight and a bigram frequency.

The manual calculations are verified using scikit-learn.

In [19]:
toy_corpus = [
    "python developer python",
    "python data analyst",
    "java developer",
    "data analyst python"
]

for i, doc in enumerate(toy_corpus, start=1):
    print(f"Document {i}: {doc}")

Document 1: python developer python
Document 2: python data analyst
Document 3: java developer
Document 4: data analyst python
